In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
# para entrega 2 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json


In [2]:
data_path = os.path.join("data")

# obtener los archivos excel
excel_files = [f for f in os.listdir(data_path) if f.endswith('.xlsx')]

dataframes = {}
for file in excel_files:
    file_path = os.path.join(data_path, file)
    # Usar el nombre del archivo sin extensión como clave
    df_name = file.replace('.xlsx', '')
    dataframes[df_name] = pd.read_excel(file_path)
    print(f"Cargado: {file}")

# mostramos los dataframes cargados
print(f"\nTotal de archivos cargados: {len(dataframes)}")
print(f"Dataframes disponibles: {list(dataframes.keys())}")


Cargado: LaLiga19 20.xlsx
Cargado: LaLiga21 22.xlsx
Cargado: LaLiga17 18.xlsx
Cargado: LaLiga20 21.xlsx
Cargado: LaLiga16 17.xlsx
Cargado: LaLiga18 19.xlsx

Total de archivos cargados: 6
Dataframes disponibles: ['LaLiga19 20', 'LaLiga21 22', 'LaLiga17 18', 'LaLiga20 21', 'LaLiga16 17', 'LaLiga18 19']


In [3]:
dataframes.keys()

dict_keys(['LaLiga19 20', 'LaLiga21 22', 'LaLiga17 18', 'LaLiga20 21', 'LaLiga16 17', 'LaLiga18 19'])

In [4]:
display(dataframes['LaLiga19 20'].head())

,Wk,Day,Date,Time,Local,xG,Score,xG.1,Visitante,Attendance,Venue,Referee,Informe del partido,Notes
0,1.0,Vie,2019-08-16,21:00 (15:00),Athletic Club,0.5,1–0,0.9,Barcelona,47.693,San Mamés,Carlos del Cerro,Informe del partido,NaN
1,1.0,Sáb,2019-08-17,17:00 (11:00),Celta Vigo,0.8,1–3,1.5,Real Madrid,23.566,Estadio de Balaídos,Javier Estrada,Informe del partido,NaN
2,1.0,Sáb,2019-08-17,19:00 (13:00),Valencia,2.3,1–1,1.3,Real Sociedad,41.846,Estadio de Mestalla,Jesús Gil,Informe del partido,NaN
3,1.0,Sáb,2019-08-17,20:00 (14:00),Mallorca,1.7,2–1,0.7,Eibar,15.127,Iberostar Estadi,Mario Melero,Informe del partido,NaN
4,1.0,Sáb,2019-08-17,21:00 (15:00),Villarreal,1.6,4–4,2.2,Granada,14.753,Estadio de la Cerámica,Adrián Cordero,Informe del partido,NaN


In [5]:
equipos = ['Real Madrid', 'Barcelona']
columnas_a_borrar = ['Day', 'Time', 'xG', 'xG.1', 'Referee', 'Informe del partido', 'Notes']

dataframes_filtrados = {}

for key, df in dataframes.items():
    filtro = df['Local'].isin(equipos) | df['Visitante'].isin(equipos) #que el local o el visitante esté en los equipos
    df_filtrado = df[filtro]
    df_limpio = df_filtrado.drop(columns=columnas_a_borrar, errors='ignore') #eliminar las columnas que no son de interes
    dataframes_filtrados[key] = df_limpio #guardo el cambio

# verificamos que el dataframe este ok
display(dataframes_filtrados['LaLiga19 20'].head())

,Wk,Date,Local,Score,Visitante,Attendance,Venue
0,1.0,2019-08-16,Athletic Club,1–0,Barcelona,47.693,San Mamés
1,1.0,2019-08-17,Celta Vigo,1–3,Real Madrid,23.566,Estadio de Balaídos
14,2.0,2019-08-24,Real Madrid,1–1,Valladolid,63.037,Estadio Santiago Bernabéu
20,2.0,2019-08-25,Barcelona,5–2,Betis,79.159,Camp Nou
24,3.0,2019-08-31,Osasuna,2–2,Barcelona,16.742,Estadio El Sadar


In [6]:
lista_goles_por_año = []

equipos = ['Real Madrid', 'Barcelona']

for temporada, df in dataframes_filtrados.items():
    df_temp = df.copy()
    df_temp[['Goles_Local', 'Goles_Visitante']] = df_temp['Score'].astype(str).str.split(r'[-–]', expand=True).astype(float)
    
    datos_equipos = []
    
    # ciclo para calcular goles por equipo
    for equipo in equipos:
        goles_local = df_temp[df_temp['Local'] == equipo]['Goles_Local'].sum()
        goles_visita = df_temp[df_temp['Visitante'] == equipo]['Goles_Visitante'].sum()
        datos_equipos.append({
            'Temporada': temporada,
            'Equipo': equipo,
            'Goles Local': int(goles_local),
            'Goles Visita': int(goles_visita),
            'Goles Totales': int(goles_local + goles_visita)
        })

    df_resumen_año = pd.DataFrame(datos_equipos)
    lista_goles_por_año.append(df_resumen_año)

print("Resumen de la primera temporada procesada:")
for df in lista_goles_por_año:
    print('='*62)
    display(df)

Resumen de la primera temporada procesada:


,Temporada,Equipo,Goles Local,Goles Visita,Goles Totales
0,LaLiga19 20,Real Madrid,40,30,70
1,LaLiga19 20,Barcelona,52,34,86


,Temporada,Equipo,Goles Local,Goles Visita,Goles Totales
0,LaLiga21 22,Real Madrid,44,36,80
1,LaLiga21 22,Barcelona,37,31,68


,Temporada,Equipo,Goles Local,Goles Visita,Goles Totales
0,LaLiga17 18,Real Madrid,54,40,94
1,LaLiga17 18,Barcelona,53,46,99


,Temporada,Equipo,Goles Local,Goles Visita,Goles Totales
0,LaLiga20 21,Real Madrid,33,34,67
1,LaLiga20 21,Barcelona,44,41,85


,Temporada,Equipo,Goles Local,Goles Visita,Goles Totales
0,LaLiga16 17,Real Madrid,48,58,106
1,LaLiga16 17,Barcelona,64,52,116


,Temporada,Equipo,Goles Local,Goles Visita,Goles Totales
0,LaLiga18 19,Real Madrid,32,31,63
1,LaLiga18 19,Barcelona,51,39,90


Agregamos una funcion útil para insertar los logos de cada equipo.

In [7]:
def agregar_logo(img, x, y, ax, zoom=0.025):
    imagebox = OffsetImage(img, zoom=zoom)
    # con xybox=(-35, 0) ponemos el logo a la izquierda del punto para no taparlo
    ab = AnnotationBbox(imagebox, (x, y), xybox=(-35, 0), xycoords='data', boxcoords='offset points', frameon=False)
    ax.add_artist(ab)

In [8]:
# ── Paso 1: Consolidar y ordenar ──────────────────────────────────────────────
df_goles = pd.concat(lista_goles_por_año, ignore_index=True)

# Formatear la temporada: "LaLiga19 20" → "19-20"
df_goles['Temporada_Corta'] = (
    df_goles['Temporada']
    .str.replace('LaLiga', '', regex=False)
    .str.strip()
    .str.replace(' ', '-')
)

# Ordenar cronológicamente por temporada
df_goles = df_goles.sort_values('Temporada_Corta').reset_index(drop=True)

# ── Paso 2: Calcular promedios (estaban ausentes del DF) ──────────────────────
PARTIDOS_POR_EQUIPO = 19  # cada equipo juega 19 partidos como local en una liga de 20
df_goles['Promedio Local']  = (df_goles['Goles Local']  / PARTIDOS_POR_EQUIPO).round(2)
df_goles['Promedio Visita'] = (df_goles['Goles Visita'] / PARTIDOS_POR_EQUIPO).round(2)

# ── Paso 3: Construir el diccionario para la web ───────────────────────────────
datos_web = {"realMadrid": [], "barcelona": []}

for _, row in df_goles.iterrows():
    entrada = {
        "temporada":      row['Temporada_Corta'],        # ej. "19-20"
        "local":          int(row['Goles Local']),
        "visita":         int(row['Goles Visita']),
        "partidos":       PARTIDOS_POR_EQUIPO,
        "promedioLocal":  float(row['Promedio Local']),
        "promedioVisita": float(row['Promedio Visita']),
    }

    if row['Equipo'] == 'Real Madrid':
        datos_web["realMadrid"].append(entrada)
    elif row['Equipo'] == 'Barcelona':
        datos_web["barcelona"].append(entrada)

# ── Paso 4: Exportar JSON ──────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)   # exist_ok evita el crash si la carpeta ya existe

output_path = os.path.join('data', 'data.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(datos_web, f, indent=4, ensure_ascii=False)

# ── Verificación rápida ────────────────────────────────────────────────────────
print(f"✅  Exportado → {output_path}")
print(f"    Real Madrid : {len(datos_web['realMadrid'])} temporadas")
print(f"    Barcelona   : {len(datos_web['barcelona'])} temporadas")
print()
print("Vista previa del JSON generado:")
print(json.dumps(datos_web, indent=4, ensure_ascii=False)[:800], "...")

✅  Exportado → data/data.json
    Real Madrid : 6 temporadas
    Barcelona   : 6 temporadas

Vista previa del JSON generado:
{
    "realMadrid": [
        {
            "temporada": "16-17",
            "local": 48,
            "visita": 58,
            "partidos": 19,
            "promedioLocal": 2.53,
            "promedioVisita": 3.05
        },
        {
            "temporada": "17-18",
            "local": 54,
            "visita": 40,
            "partidos": 19,
            "promedioLocal": 2.84,
            "promedioVisita": 2.11
        },
        {
            "temporada": "18-19",
            "local": 32,
            "visita": 31,
            "partidos": 19,
            "promedioLocal": 1.68,
            "promedioVisita": 1.63
        },
        {
            "temporada": "19-20",
            "local": 40,
            "visita": 30,
            "partidos": 19,
            "promedioLocal": 2.11,
         ...


In [11]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

# 1. Funciones y configuraciones base
def categorizar_goles(goles):
    if pd.isna(goles): return None
    goles = int(goles)
    if goles == 0: return '0 goles'
    elif goles == 1: return '1 gol'
    elif goles == 2: return '2 goles'
    else: return '3+ goles'

orden_categorias = ['0 goles', '1 gol', '2 goles', '3+ goles']
equipos = ['Real Madrid', 'Barcelona']
colores = {
    'Real Madrid': {'Local': '#005A9F', 'Visita': '#00a8ff'}, 
    'Barcelona': {'Local': '#A50044', 'Visita': '#004D98'}
}

In [12]:
# === GRÁFICO TEMPORADA 16-17 ===
df_actual = dataframes_filtrados['LaLiga16 17'].copy()
df_actual[['Goles_Local', 'Goles_Visitante']] = df_actual['Score'].astype(str).str.split(r'[-–]', expand=True).astype(float)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Real Madrid", "FC Barcelona"))

for i, equipo in enumerate(equipos, 1):
    goles_local = df_actual[df_actual['Local'] == equipo]['Goles_Local'].apply(categorizar_goles)
    dist_local = goles_local.value_counts().reindex(orden_categorias, fill_value=0)
    
    goles_visita = df_actual[df_actual['Visitante'] == equipo]['Goles_Visitante'].apply(categorizar_goles)
    dist_visita = goles_visita.value_counts().reindex(orden_categorias, fill_value=0)
    
    fig.add_trace(go.Bar(x=dist_local.index, y=dist_local.values, name=f'Local', marker_color=colores[equipo]['Local'], offsetgroup=0), row=1, col=i)
    fig.add_trace(go.Bar(x=dist_visita.index, y=dist_visita.values, name=f'Visita', marker_color=colores[equipo]['Visita'], offsetgroup=1), row=1, col=i)

fig.update_layout(
    title_text="Consistencia Ofensiva: Temporada 16-17",
    barmode='group', template='plotly_white', 
    height=450, width=850, bargap=0.2, # <-- Esto evita que se estire
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5)
)
fig.update_yaxes(title_text="Cantidad de Partidos", row=1, col=1)
fig.show()

In [ ]:
# === GRÁFICOS TEMPORADA 16-17 ===
df_actual = dataframes_filtrados['LaLiga16 17'].copy()
df_actual[['Goles_Local', 'Goles_Visitante']] = df_actual['Score'].astype(str).str.split(r'[-–]', expand=True).astype(float)

for equipo in equipos:
    goles_local = df_actual[df_actual['Local'] == equipo]['Goles_Local'].apply(categorizar_goles)
    dist_local = goles_local.value_counts().reindex(orden_categorias, fill_value=0)
    
    goles_visita = df_actual[df_actual['Visitante'] == equipo]['Goles_Visitante'].apply(categorizar_goles)
    dist_visita = goles_visita.value_counts().reindex(orden_categorias, fill_value=0)
    
    fig = go.Figure()
    fig.add_trace(go.Bar(x=dist_local.index, y=dist_local.values, name='Local', marker_color=colores[equipo]['Local']))
    fig.add_trace(go.Bar(x=dist_visita.index, y=dist_visita.values, name='Visita', marker_color=colores[equipo]['Visita']))
    
    fig.update_layout(
        title_text=f"{equipo} - Consistencia Ofensiva (16-17)",
        barmode='group', template='plotly_white', 
        height=400, width=500, bargap=0.2, # Ancho reducido para que no se estire
        yaxis_title="Cantidad de Partidos",
        legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
    )
    fig.show()
    